# 02 — Transformer Architecture
This notebook implements an Encoder-only Transformer utilizing multi-head self-attention.
It replaces recurrence with attention to capture long-range dependencies across the historical window.

In [1]:
import os, sys, numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, os.path.abspath('..'))
DATA_PATH='../data'; SEQ_LEN=60; FORECAST=5; NUM_STOCKS=50; TRAIN_R=0.8; TGT=1; BATCH=64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Ready | device:', device)


Ready | device: cpu


## Data

In [2]:
def make_seq(data):
    X,y=[],[]
    for i in range(len(data)-SEQ_LEN-FORECAST+1):
        X.append(data[i:i+SEQ_LEN])
        y.append(data[i+SEQ_LEN:i+SEQ_LEN+FORECAST, TGT])
    return np.array(X), np.array(y)

def load_data():
    files=[f for f in sorted(os.listdir(DATA_PATH)) if f.endswith('.txt')][:NUM_STOCKS]
    dfs=[]
    for f in files:
        try:
            df=pd.read_csv(os.path.join(DATA_PATH,f))
            if not {'Date','Close'}.issubset(df.columns): continue
            df['Date']=pd.to_datetime(df['Date']); df=df.sort_values('Date')
            df['Return']=df['Close'].pct_change(); df['MA_10']=df['Close'].rolling(10).mean()
            df['MA_50']=df['Close'].rolling(50).mean(); df=df.dropna(); df['Stock']=f
            if len(df)>SEQ_LEN+FORECAST: dfs.append(df)
        except: pass
    return pd.concat(dfs,ignore_index=True)

cdf=load_data()
all_X,all_y=[],[]
for s in cdf['Stock'].unique():
    sdf=cdf[cdf['Stock']==s]; sc=MinMaxScaler()
    d=sc.fit_transform(sdf[['Close','Return','MA_10','MA_50']]); Xs,ys=make_seq(d)
    if len(Xs): all_X.append(Xs); all_y.append(ys)
X=np.concatenate(all_X); y=np.concatenate(all_y)
sp=int(TRAIN_R*len(X)); X_tr,X_te=X[:sp],X[sp:]; y_tr,y_te=y[:sp],y[sp:]
T=lambda a: torch.tensor(a,dtype=torch.float32).to(device)
Xt,Xe,yt_t,ye_t=T(X_tr),T(X_te),T(y_tr),T(y_te)
print(f'Train:{len(X_tr):,} | Test:{len(X_te):,} | Shape X:{X.shape}')


Train:108,644 | Test:27,161 | Shape X:(135805, 60, 4)


## Model — Transformer

In [3]:
from src.transformer_model import StockTransformer as TransformerModel


## Training

In [4]:
MODEL_NAME='Transformer'; EPOCHS=25; LR=3e-4
model=TransformerModel(input_size=4).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
ldr=DataLoader(TensorDataset(Xt,yt_t),batch_size=BATCH,shuffle=True)
losses=[]
for ep in range(EPOCHS):
    model.train(); tot=0
    for xb,yb in ldr:
        p=model(xb); loss=nn.MSELoss()(p,yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step(); tot+=loss.item()
    avg=tot/len(ldr); losses.append(avg); sch.step()
    print(f'Epoch {ep+1:02d}/{EPOCHS} MSE:{avg:.6f}')
torch.save(model.state_dict(),'../models/transformer_model.pth')
model.eval()
with torch.no_grad(): mu=model(Xe).cpu().numpy()


Params: 596,101


/Users/shruthirajgangapuri/deep_learning_stock_project/src/transformer_model.py:114: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch 01/25 MSE:0.004781


Epoch 02/25 MSE:0.003566


Epoch 03/25 MSE:0.003508


...

Epoch 21/25 MSE:0.003307


Epoch 22/25 MSE:0.003302


Epoch 23/25 MSE:0.003300


Epoch 24/25 MSE:0.003297


Epoch 25/25 MSE:0.003296


## Save Results

In [5]:
os.makedirs('../outputs',exist_ok=True); os.makedirs('../models',exist_ok=True)
import pandas as pd
rows=[]
for d in range(FORECAST):
    mae=mean_absolute_error(y_te[:,d],mu[:,d]); rmse=float(np.sqrt(mean_squared_error(y_te[:,d],mu[:,d])))
    rows.append({'model':MODEL_NAME,'day':f'Day+{d+1}','mae':round(mae,6),'rmse':round(rmse,6)})
ov_mae=mean_absolute_error(y_te.flatten(),mu.flatten()); ov_rmse=float(np.sqrt(mean_squared_error(y_te.flatten(),mu.flatten())))
rows.append({'model':MODEL_NAME,'day':'Overall','mae':round(ov_mae,6),'rmse':round(ov_rmse,6)})
rp='../outputs/results.csv'; new=pd.DataFrame(rows)
if os.path.exists(rp):
    ex=pd.read_csv(rp); ex=ex[ex['model']!=MODEL_NAME]; new=pd.concat([ex,new],ignore_index=True)
new.to_csv(rp,index=False); print(f'Saved {MODEL_NAME} → {rp}')
print(new[new['model']==MODEL_NAME].to_string(index=False))


Saved Transformer → ../outputs/results.csv
      model     day      mae     rmse  avg_uncertainty
Transformer   Day+1 0.042501 0.061532              NaN
Transformer   Day+2 0.042574 0.061670              NaN
Transformer   Day+3 0.042312 0.061550              NaN
Transformer   Day+4 0.042727 0.061899              NaN
Transformer   Day+5 0.042880 0.062001              NaN
Transformer Overall 0.042599 0.061731              NaN


## Plots

In [6]:
plt.figure(figsize=(8,4)); plt.plot(losses,color='#9B59B6',lw=2)
plt.title('Transformer Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.tight_layout(); plt.savefig('../outputs/transformer_loss.png',dpi=150); plt.close()
n=min(300,len(y_te)); idx=np.arange(n)
plt.figure(figsize=(14,4))
plt.plot(idx,y_te[:n,0],label='Actual',color='#2ECC71',lw=1.2)
plt.plot(idx,mu[:n,0],label='Predicted',color='#9B59B6',lw=1.2,alpha=0.85)
plt.title('Transformer: Actual vs Predicted Return — Day+1'); plt.xlabel('Sample'); plt.ylabel('Scaled Return')
plt.legend(); plt.tight_layout(); plt.savefig('../outputs/transformer_pred.png',dpi=150); plt.close()
print('Plots saved')


Plots saved
